# Exploratory Data Analysis

Explore distributions, trends, relationships, and air quality patterns.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Load the processed dataset here.

In [ ]:
con = sqlite3.connect("../data/airlens.db")

df = pd.read_sql_query(
    "SELECT * FROM air_quality_validated",
    con
)

con.close()

df["Date"] = pd.to_datetime(df["Date"])

df.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,AQI,AQI_Bucket,Valid_AQI,AQI_Outlier
0,Ahmedabad,2015-01-01,NaN,NaN,0.92,18.22,17.15,NaN,0.92,27.64,133.36,NaN,NaN,NaN,0
1,Ahmedabad,2015-01-02,NaN,NaN,0.97,15.69,16.46,NaN,0.97,24.55,34.06,NaN,NaN,NaN,0
2,Ahmedabad,2015-01-03,NaN,NaN,17.40,19.30,29.70,NaN,17.40,29.07,30.70,NaN,NaN,NaN,0
3,Ahmedabad,2015-01-04,NaN,NaN,1.70,18.48,17.97,NaN,1.70,18.59,36.08,NaN,NaN,NaN,0
4,Ahmedabad,2015-01-05,NaN,NaN,22.10,21.42,37.76,NaN,22.10,39.33,39.31,NaN,NaN,NaN,0


In [ ]:
df.shape

(29531, 15)

In [ ]:
df["Valid_AQI"].describe()

count    24307.000000
mean       153.345909
std        102.323170
min         13.000000
25%         80.000000
50%        116.000000
75%        197.000000
max        500.000000
Name: Valid_AQI, dtype: float64

In [ ]:
#Define reliable cities
# We'll reuse our earlier data-quality logic.

city_quality = (
    df.groupby("City")
    .agg(
        total_records=("Date", "count"),
        valid_aqi=("Valid_AQI", "count"),
        average_aqi=("Valid_AQI", "mean")
    )
)

city_quality["coverage"] = (
    city_quality["valid_aqi"] /
    city_quality["total_records"] * 100
)

reliable_cities = city_quality[
    (city_quality["valid_aqi"] >= 500) &
    (city_quality["coverage"] >= 80)
].index.tolist()

reliable_cities

['Amaravati',
 'Amritsar',
 'Bengaluru',
 'Chennai',
 'Delhi',
 'Gurugram',
 'Hyderabad',
 'Jaipur',
 'Kolkata',
 'Lucknow',
 'Thiruvananthapuram',
 'Visakhapatnam']

In [8]:
reliable_df = df[df["City"].isin(reliable_cities)].copy()
#For country-wide analytics we can still use all cities,
# but city comparisons should preferably use reliable_df.

In [ ]:

# AQI distribution
# First understand how AQI values are distributed.

fig = px.histogram(
    reliable_df,
    x="Valid_AQI",
    nbins=50,
    title="Distribution of AQI Across Reliable Cities",
    labels={"Valid_AQI": "AQI"}
)

fig.show()

# Are most air-quality observations healthy, moderate, or highly polluted?

In [10]:
#Average AQI by city
city_aqi = (
    reliable_df.groupby("City")["Valid_AQI"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

city_aqi.columns = ["City", "Average_AQI"]

city_aqi

,City,Average_AQI
0,Delhi,252.105074
1,Gurugram,219.407407
2,Lucknow,215.193823
3,Kolkata,140.566313
4,Jaipur,133.679159
5,Amritsar,117.821747
6,Visakhapatnam,117.269855
7,Chennai,114.502654
8,Hyderabad,108.155650
9,Amaravati,95.299643


In [11]:
fig = px.bar(
    city_aqi,
    x="City",
    y="Average_AQI",
    title="Average AQI by City"
)

fig.show()

In [12]:
# AQI category distribution
# Instead of depending on the original AQI_Bucket,
# derive categories ourselves from Valid_AQI.

bins = [-1, 50, 100, 200, 300, 400, 500]

labels = [
    "Good",
    "Satisfactory",
    "Moderate",
    "Poor",
    "Very Poor",
    "Severe"
]

reliable_df["AQI_Category"] = pd.cut(
    reliable_df["Valid_AQI"],
    bins=bins,
    labels=labels
)

In [13]:
reliable_df["AQI_Category"].value_counts()

AQI_Category
Satisfactory    6222
Moderate        6147
Poor            1825
Very Poor       1516
Good             868
Severe           386
Name: count, dtype: int64

In [14]:
category_counts = (
    reliable_df["AQI_Category"]
    .value_counts()
    .reindex(labels)
    .reset_index()
)

category_counts.columns = ["AQI_Category", "Days"]

fig = px.bar(
    category_counts,
    x="AQI_Category",
    y="Days",
    title="AQI Category Distribution"
)

fig.show()

In [15]:
# Seasonal analysis
# Create the season column:

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    return "Post-Monsoon"

reliable_df["Season"] = reliable_df["Date"].dt.month.apply(get_season)

In [16]:
season_aqi = (
    reliable_df.groupby(["City", "Season"])["Valid_AQI"]
    .mean()
    .reset_index()
)

season_aqi.head()

,City,Season,Valid_AQI
0,Amaravati,Monsoon,65.261261
1,Amaravati,Post-Monsoon,103.410000
2,Amaravati,Summer,71.575397
3,Amaravati,Winter,139.629213
4,Amritsar,Monsoon,92.430939


In [17]:
fig = px.bar(
    season_aqi,
    x="City",
    y="Valid_AQI",
    color="Season",
    barmode="group",
    title="Seasonal AQI Comparison Across Cities",
    labels={"Valid_AQI": "Average AQI"}
)

fig.show()

In [18]:
# Pollution .trend over time
# Let's first make a monthly trend

monthly = (
    reliable_df
    .set_index("Date")
    .groupby("City")["Valid_AQI"]
    .resample("ME")
    .mean()
    .reset_index()
)

In [19]:
delhi = monthly[monthly["City"] == "Delhi"]

fig = px.line(
    delhi,
    x="Date",
    y="Valid_AQI",
    title="Delhi Monthly AQI Trend",
    labels={"Valid_AQI": "Average AQI"}
)

fig.show()

In [20]:
# Compare multiple cities
# A useful visual:

selected = [
    "Delhi",
    "Bengaluru",
    "Hyderabad",
    "Chennai",
    "Kolkata"
]

comparison = monthly[
    monthly["City"].isin(selected)
]

fig = px.line(
    comparison,
    x="Date",
    y="Valid_AQI",
    color="City",
    title="AQI Trend Comparison Across Major Cities"
)

fig.show()

In [21]:
# Pollutant correlation
# Now one of the most important EDA sections.

pollutants = [
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Valid_AQI"
]

corr = reliable_df[pollutants].corr()

corr.round(2)

,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Valid_AQI
PM2.5,1.00,0.86,0.51,0.45,0.48,0.29,0.18,0.26,0.21,0.89
PM10,0.86,1.00,0.69,0.64,0.62,0.54,0.32,0.42,0.32,0.91
NO,0.51,0.69,1.00,0.66,0.78,0.23,0.07,0.27,0.12,0.56
NO2,0.45,0.64,0.66,1.00,0.72,0.19,0.04,0.30,0.27,0.52
NOx,0.48,0.62,0.78,0.72,1.00,0.19,0.07,0.23,0.20,0.54
NH3,0.29,0.54,0.23,0.19,0.19,1.00,0.11,0.11,0.08,0.28
CO,0.18,0.32,0.07,0.04,0.07,0.11,1.00,0.14,0.00,0.29
SO2,0.26,0.42,0.27,0.30,0.23,0.11,0.14,1.00,0.18,0.31
O3,0.21,0.32,0.12,0.27,0.20,0.08,0.00,0.18,1.00,0.30
Valid_AQI,0.89,0.91,0.56,0.52,0.54,0.28,0.29,0.31,0.30,1.00


In [ ]:
fig = px.imshow(
    corr,
    text_auto=".2f", # type: ignore
    title="Pollutant Correlation Matrix"
)

fig.show()